# Baptista et al. §5.4 — rectangles, U-Net, $N=2$: direct reproduction

Reproduces the memorization experiment of Baptista, Dasgupta, Kovachki, Oberai & Stuart,
*Memorization and Regularization in Generative Diffusion Models*,
[arXiv:2501.15785](https://arxiv.org/abs/2501.15785), **§5.4** — Figures 16, 17 and the
**left panel of Figure 18** ($N=2$).

This notebook is a **reproduction of their setup**, not an arm of our multiband Matérn study.
Nothing here is comparable to `edm_unet_*` numbers and none of the repo's locked conventions
(σ_max = 10, the per-band ring metric, `exclude_nn`) apply — this uses *their* sampler, *their*
architecture and *their* pixel-space collapse metric throughout. It is a standalone
positive control: **can a faithful copy of their pipeline memorize?**

## Provenance of every setting

Two sources, used in this order of authority:

1. **Their released code** — [`baptistar/DiffusionModelDynamics`](https://github.com/baptistar/DiffusionModelDynamics),
   `RectangleImages/main.py` (single commit `2719b5d`). This is a fork of
   [NVlabs/edm](https://github.com/NVlabs/edm) (Karras et al. 2022, the paper's ref [19]);
   `training/networks.py`, `training/loss.py` and `generate.py` are byte-identical to upstream
   apart from two commented-out debug lines. Everything `main.py` specifies is taken **verbatim**.
2. **The paper text of §5.4**, for the Figure-18 protocol only — the model-size sweep script is
   *not* in the released repo, so the six-model sweep, the 100-sample evaluation and the
   thresholded exact-match metric come from the prose.

| | setting | source |
|---|---|---|
| data | $64\times64$, $\{0,1\}$, $d=4096$; `data[0,0,6:18,6:18]=1`, `data[1,0,40:54,40:54]=1` | `main.py:27-29` (verbatim) |
| network | `EDMPrecond(SongUNet)`, `img_resolution=64`, `img_channels=1`, `label_dim=0`, `model_channels=128`, `channel_mult=[2,2,2]`, `num_blocks=4`, `attn_resolutions=[16]`, `embedding_type='positional'`, `channel_mult_noise=1`, `resample_filter=[1,1]`, `encoder_type='standard'`, `decoder_type='standard'`, `dropout=0.0`, `use_fp16=False` | `main.py:50-57` |
| $\sigma_{data}$ | **0.5** (EDM default, not estimated from the data) | `networks.py:639` |
| loss | `EDMLoss`, `P_mean=-1.2`, `P_std=1.2`, $\sigma_{data}=0.5$; reduced as `loss.sum()/B` | `loss.py:65`, `main.py:84` |
| optimizer | Adam, `lr=10e-4`, `betas=[0.9,0.999]`, `eps=1e-8` | `main.py:61` |
| **lr schedule** | `lr = 10e-4 * min(cur_nimg / 1e7, 1)` — **see the warning below** | `main.py:19,89` |
| EMA | `ema_halflife_kimg=500`, `ema_rampup_ratio=0.05`; **samples are drawn from the EMA weights** | `main.py:20-21,96-102,131` |
| grad guard | `torch.nan_to_num(grad, nan=0, posinf=1e5, neginf=-1e5)` | `main.py:91-93` |
| batching | `DataLoader(TensorDataset(data), batch_size=1, shuffle=True)` — **see the warning below** | `main.py:37` |
| duration | `epochs = 50000` | `main.py:15` |
| sampler | `edm_sampler(ema, latents, num_steps=40)`, all other args at EDM defaults | `main.py:134` |
| metric | binarize generated image at 0.5, count exact matches (Euclidean distance 0) among 100 samples, every 1000 optimization steps | paper §5.4 |
| model sweep | 6 sizes, `model_channels` ∈ {4, 8, 16, 32, 64, 128} | derived, verified exactly — see below |

## Three things in their setup that are easy to get wrong

**1. The learning rate never reaches $10^{-3}$.** `main.py:89` applies EDM's ramp-up,
`lr = 10\mathrm{e}{-4}\cdot\min(\texttt{cur\_nimg}/10^7,\,1)`, and `cur_nimg` advances by the
batch size (1) per update. Over the entire 50,000-epoch run `cur_nimg` reaches only ~$10^5$, so
the ramp factor never exceeds **0.01** and the effective learning rate rises linearly from
~0 to $10^{-5}$ — it is in permanent ramp-up. This is inherited from EDM's default
`lr_rampup_kimg=10000`, which is calibrated for 200-Mimg runs. It is not a typo on their part but
it *is* load-bearing: training at a flat $10^{-3}$ is a ~100× larger step size and produces
completely different memorization timing. Reproduced exactly here; `CFG['lr_rampup_kimg']` exposes it.

**2. "Batch size 2" is a `DataLoader` with `batch_size=1`.** `main.py:16` defines `bsize = 2`
and then never uses it; line 37 builds the loader with `batch_size=1`. With $N=2$ that means each
epoch is a shuffled pass doing **two** optimizer updates on **one** image each — not one update on
a batch of two. The gradient is therefore per-sample, and 50,000 epochs is 100,000 optimizer
steps. `CFG['batch_mode']` selects `'main_py'` (their code, the default) or `'paper_bs2'`
(one update per epoch on a batch of 2, which is what the prose describes).

**3. As released, the sampler is the deterministic ODE, not an SDE.** §5.4 says *"The reverse
process for sampling is the SDE-based methodology presented in [19]"* and *"By using an SDE for
data generation, we demonstrate that our memorization results, so far all in the ODE setting, are
also empirically verified in the SDE setting."* But `main.py:134` calls
`edm_sampler(ema, latents, num_steps=40)` without overriding `S_churn`, whose default in
`generate.py:28` is **0**. With `S_churn=0` the churn factor `gamma` is 0, so `t_hat == t_cur`,
the injected-noise term is scaled by $\sqrt{t_{hat}^2-t_{cur}^2}=0$, and Algorithm 2 degenerates
to deterministic 2nd-order Heun. (The same zero makes their `edm_sampler_fixednoise` variant
mathematically identical to `edm_sampler` — the "fixed noise increments" of the Figure 17 caption
have no effect at `S_churn=0`.) Either they passed churn at runtime without committing it, or the
released code does not match the paper's SDE claim. **This is a discrepancy in their material, not
a decision this notebook should make silently.** `CFG['S_churn']` defaults to **0** — the literal
released code — and `CFG_SDE_ALT` below holds Karras's CIFAR-10 stochastic preset
(`S_churn=30, S_min=0.01, S_max=1, S_noise=1.007`) for a one-line switch.

## The six model sizes

Figure 18's legend gives 57017 / 222705 / 880097 / 3498945 / 13952897 / 55725825 parameters,
and the caption says they come from *"varying the number of channels in the first residual
block"*. Holding everything else at `main.py`'s network config and sweeping `model_channels`
reproduces **all six counts exactly** (asserted in a cell below):

| `model_channels` | 4 | 8 | 16 | 32 | 64 | 128 |
|---|---|---|---|---|---|---|
| parameters | 57,017 | 222,705 | 880,097 | 3,498,945 | 13,952,897 | 55,725,825 |

`model_channels=128` is `main.py`'s own value, i.e. the largest arm is their released config.

## What this notebook asserts

**Gate: the collapse fraction must reach ~1.0 at $N=2$ for every model size**, sooner for larger
models. That is the claim of Figure 18 left. If it does not reproduce, the discrepancy is in this
transcription, not in our Matérn results.

## Setup

`src/` is used for `device_utils` and `project_paths` only, plus a numerical cross-check of the
transcribed EDM preconditioner against `src/edm.py`. No file in `src/` is modified or needed to
run this notebook, so every previously committed result is untouched.

In [1]:
import os, sys, math, time, copy, json
import numpy as np
import torch
import matplotlib

# Headless-safe: picks Agg under SLURM (no $DISPLAY), leaves inline alone in Jupyter.
if not os.environ.get('DISPLAY') and not hasattr(sys, 'ps1'):
    matplotlib.use('Agg')
import matplotlib.pyplot as plt

# -- path setup (same pattern as the other notebooks in this folder) --
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
src_dir = os.path.join(repo_root, 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from device_utils import resolve_device

DEVICE = resolve_device()           # cuda > mps > cpu
torch.backends.cudnn.benchmark = True

# Heavy artifacts (resume state is ~0.9 GB for the 55.7M arm) can be redirected off a
# shared project quota so a big run cannot squeeze anyone else in the allocation:
#     export RECT_RESULTS_DIR=$SCRATCH/rectangles_n2
results_dir = os.environ.get('RECT_RESULTS_DIR',
                             os.path.join(repo_root, 'results', 'data'))
fig_dir = os.environ.get('RECT_FIG_DIR',
                         os.path.join(repo_root, 'results', 'figures'))
os.makedirs(results_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)

print(f'torch {torch.__version__}')
print(f'results -> {results_dir}')
print(f'figures -> {fig_dir}')
print(f'device: {DEVICE}')
if DEVICE.type == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print(f'gpu: {p.name}, {p.total_memory / 1e9:.1f} GB, cc {p.major}.{p.minor}')
elif DEVICE.type != 'cuda':
    print('WARNING: this notebook is sized for a CUDA GPU. The 55.7M-parameter arm is not '
          'practical on CPU/MPS -- set CFG["model_channels_sweep"] to the small sizes, or '
          'raise SMOKE, if you are just checking the pipeline runs.')

torch 2.13.0
results -> /home/ishiyer/projects/aip-baptista/ishiyer/research-diffusion/results/data
figures -> /home/ishiyer/projects/aip-baptista/ishiyer/research-diffusion/results/figures
device: cuda
gpu: NVIDIA L40S, 47.7 GB, cc 8.9


## The EDM network, verbatim

Transcribed from `RectangleImages/training/networks.py` in the paper's repo, which is
byte-identical to [NVlabs/edm](https://github.com/NVlabs/edm) `training/networks.py` apart from
comments. Only two mechanical changes are made, neither of which touches any computation:

* the `@persistence.persistent_class` decorators and the `torch_utils` import are dropped
  (they exist for `pickle` round-tripping of network snapshots, which this notebook does not use);
* `DhariwalUNet`, `VPPrecond`, `VEPrecond` and `iDDPMPrecond` are omitted — `main.py` uses
  `SongUNet` under `EDMPrecond` and nothing else.

`SongUNet.forward` and `EDMPrecond.forward` are unmodified, so the network, its initialization and
its preconditioning are exactly the ones that produced the paper's figures.

> Karras, Aittala, Aila & Laine, *Elucidating the Design Space of Diffusion-Based Generative
> Models*, NeurIPS 2022. Code © 2022 NVIDIA CORPORATION & AFFILIATES, released under
> CC BY-NC-SA 4.0. Reproduced here for research use.

In [2]:
# ---------------------------------------------------------------------------------------------
# Transcribed verbatim from RectangleImages/training/networks.py in baptistar/DiffusionModelDynamics
# (byte-identical to NVlabs/edm training/networks.py apart from comments).
#
# Copyright (c) 2022, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# Licensed under CC BY-NC-SA 4.0 -- http://creativecommons.org/licenses/by-nc-sa/4.0/
# "Elucidating the Design Space of Diffusion-Based Generative Models", Karras et al., NeurIPS 2022.
#
# Changes: @persistence decorators and the torch_utils import removed (pickle plumbing only);
# DhariwalUNet / VPPrecond / VEPrecond / iDDPMPrecond omitted (unused by main.py).
# No computational change.
# ---------------------------------------------------------------------------------------------

from torch.nn.functional import silu

def weight_init(shape, mode, fan_in, fan_out):
    if mode == 'xavier_uniform': return np.sqrt(6 / (fan_in + fan_out)) * (torch.rand(*shape) * 2 - 1)
    if mode == 'xavier_normal':  return np.sqrt(2 / (fan_in + fan_out)) * torch.randn(*shape)
    if mode == 'kaiming_uniform': return np.sqrt(3 / fan_in) * (torch.rand(*shape) * 2 - 1)
    if mode == 'kaiming_normal':  return np.sqrt(1 / fan_in) * torch.randn(*shape)
    raise ValueError(f'Invalid init mode "{mode}"')

#----------------------------------------------------------------------------
# Fully-connected layer.

class Linear(torch.nn.Module):
    def __init__(self, in_features, out_features, bias=True, init_mode='kaiming_normal', init_weight=1, init_bias=0):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        init_kwargs = dict(mode=init_mode, fan_in=in_features, fan_out=out_features)
        self.weight = torch.nn.Parameter(weight_init([out_features, in_features], **init_kwargs) * init_weight)
        self.bias = torch.nn.Parameter(weight_init([out_features], **init_kwargs) * init_bias) if bias else None

    def forward(self, x):
        x = x @ self.weight.to(x.dtype).t()
        if self.bias is not None:
            x = x.add_(self.bias.to(x.dtype))
        return x

#----------------------------------------------------------------------------
# Convolutional layer with optional up/downsampling.

class Conv2d(torch.nn.Module):
    def __init__(self,
        in_channels, out_channels, kernel, bias=True, up=False, down=False,
        resample_filter=[1,1], fused_resample=False, init_mode='kaiming_normal', init_weight=1, init_bias=0,
    ):
        assert not (up and down)
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.up = up
        self.down = down
        self.fused_resample = fused_resample
        init_kwargs = dict(mode=init_mode, fan_in=in_channels*kernel*kernel, fan_out=out_channels*kernel*kernel)
        self.weight = torch.nn.Parameter(weight_init([out_channels, in_channels, kernel, kernel], **init_kwargs) * init_weight) if kernel else None
        self.bias = torch.nn.Parameter(weight_init([out_channels], **init_kwargs) * init_bias) if kernel and bias else None
        f = torch.as_tensor(resample_filter, dtype=torch.float32)
        f = f.ger(f).unsqueeze(0).unsqueeze(1) / f.sum().square()
        self.register_buffer('resample_filter', f if up or down else None)

    def forward(self, x):
        w = self.weight.to(x.dtype) if self.weight is not None else None
        b = self.bias.to(x.dtype) if self.bias is not None else None
        f = self.resample_filter.to(x.dtype) if self.resample_filter is not None else None
        w_pad = w.shape[-1] // 2 if w is not None else 0
        f_pad = (f.shape[-1] - 1) // 2 if f is not None else 0

        if self.fused_resample and self.up and w is not None:
            x = torch.nn.functional.conv_transpose2d(x, f.mul(4).tile([self.in_channels, 1, 1, 1]), groups=self.in_channels, stride=2, padding=max(f_pad - w_pad, 0))
            x = torch.nn.functional.conv2d(x, w, padding=max(w_pad - f_pad, 0))
        elif self.fused_resample and self.down and w is not None:
            x = torch.nn.functional.conv2d(x, w, padding=w_pad+f_pad)
            x = torch.nn.functional.conv2d(x, f.tile([self.out_channels, 1, 1, 1]), groups=self.out_channels, stride=2)
        else:
            if self.up:
                x = torch.nn.functional.conv_transpose2d(x, f.mul(4).tile([self.in_channels, 1, 1, 1]), groups=self.in_channels, stride=2, padding=f_pad)
            if self.down:
                x = torch.nn.functional.conv2d(x, f.tile([self.in_channels, 1, 1, 1]), groups=self.in_channels, stride=2, padding=f_pad)
            if w is not None:
                x = torch.nn.functional.conv2d(x, w, padding=w_pad)
        if b is not None:
            x = x.add_(b.reshape(1, -1, 1, 1))
        return x

#----------------------------------------------------------------------------
# Group normalization.

class GroupNorm(torch.nn.Module):
    def __init__(self, num_channels, num_groups=32, min_channels_per_group=4, eps=1e-5):
        super().__init__()
        self.num_groups = min(num_groups, num_channels // min_channels_per_group)
        self.eps = eps
        self.weight = torch.nn.Parameter(torch.ones(num_channels))
        self.bias = torch.nn.Parameter(torch.zeros(num_channels))

    def forward(self, x):
        x = torch.nn.functional.group_norm(x, num_groups=self.num_groups, weight=self.weight.to(x.dtype), bias=self.bias.to(x.dtype), eps=self.eps)
        return x

#----------------------------------------------------------------------------
# Attention weight computation, i.e., softmax(Q^T * K).
# Performs all computation using FP32, but uses the original datatype for
# inputs/outputs/gradients to conserve memory.

class AttentionOp(torch.autograd.Function):
    @staticmethod
    def forward(ctx, q, k):
        w = torch.einsum('ncq,nck->nqk', q.to(torch.float32), (k / np.sqrt(k.shape[1])).to(torch.float32)).softmax(dim=2).to(q.dtype)
        ctx.save_for_backward(q, k, w)
        return w

    @staticmethod
    def backward(ctx, dw):
        q, k, w = ctx.saved_tensors
        db = torch._softmax_backward_data(grad_output=dw.to(torch.float32), output=w.to(torch.float32), dim=2, input_dtype=torch.float32)
        dq = torch.einsum('nck,nqk->ncq', k.to(torch.float32), db).to(q.dtype) / np.sqrt(k.shape[1])
        dk = torch.einsum('ncq,nqk->nck', q.to(torch.float32), db).to(k.dtype) / np.sqrt(k.shape[1])
        return dq, dk

#----------------------------------------------------------------------------
# Unified U-Net block with optional up/downsampling and self-attention.
# Represents the union of all features employed by the DDPM++, NCSN++, and
# ADM architectures.

class UNetBlock(torch.nn.Module):
    def __init__(self,
        in_channels, out_channels, emb_channels, up=False, down=False, attention=False,
        num_heads=None, channels_per_head=64, dropout=0, skip_scale=1, eps=1e-5,
        resample_filter=[1,1], resample_proj=False, adaptive_scale=True,
        init=dict(), init_zero=dict(init_weight=0), init_attn=None,
    ):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.emb_channels = emb_channels
        self.num_heads = 0 if not attention else num_heads if num_heads is not None else out_channels // channels_per_head
        self.dropout = dropout
        self.skip_scale = skip_scale
        self.adaptive_scale = adaptive_scale

        self.norm0 = GroupNorm(num_channels=in_channels, eps=eps)
        self.conv0 = Conv2d(in_channels=in_channels, out_channels=out_channels, kernel=3, up=up, down=down, resample_filter=resample_filter, **init)
        self.affine = Linear(in_features=emb_channels, out_features=out_channels*(2 if adaptive_scale else 1), **init)
        self.norm1 = GroupNorm(num_channels=out_channels, eps=eps)
        self.conv1 = Conv2d(in_channels=out_channels, out_channels=out_channels, kernel=3, **init_zero)

        self.skip = None
        if out_channels != in_channels or up or down:
            kernel = 1 if resample_proj or out_channels!= in_channels else 0
            self.skip = Conv2d(in_channels=in_channels, out_channels=out_channels, kernel=kernel, up=up, down=down, resample_filter=resample_filter, **init)

        if self.num_heads:
            self.norm2 = GroupNorm(num_channels=out_channels, eps=eps)
            self.qkv = Conv2d(in_channels=out_channels, out_channels=out_channels*3, kernel=1, **(init_attn if init_attn is not None else init))
            self.proj = Conv2d(in_channels=out_channels, out_channels=out_channels, kernel=1, **init_zero)

    def forward(self, x, emb):
        orig = x
        x = self.conv0(silu(self.norm0(x)))

        params = self.affine(emb).unsqueeze(2).unsqueeze(3).to(x.dtype)
        if self.adaptive_scale:
            scale, shift = params.chunk(chunks=2, dim=1)
            x = silu(torch.addcmul(shift, self.norm1(x), scale + 1))
        else:
            x = silu(self.norm1(x.add_(params)))

        x = self.conv1(torch.nn.functional.dropout(x, p=self.dropout, training=self.training))
        x = x.add_(self.skip(orig) if self.skip is not None else orig)
        x = x * self.skip_scale

        if self.num_heads:
            q, k, v = self.qkv(self.norm2(x)).reshape(x.shape[0] * self.num_heads, x.shape[1] // self.num_heads, 3, -1).unbind(2)
            w = AttentionOp.apply(q, k)
            a = torch.einsum('nqk,nck->ncq', w, v)
            x = self.proj(a.reshape(*x.shape)).add_(x)
            x = x * self.skip_scale
        return x

#----------------------------------------------------------------------------
# Timestep embedding used in the DDPM++ and ADM architectures.

class PositionalEmbedding(torch.nn.Module):
    def __init__(self, num_channels, max_positions=10000, endpoint=False):
        super().__init__()
        self.num_channels = num_channels
        self.max_positions = max_positions
        self.endpoint = endpoint

    def forward(self, x):
        freqs = torch.arange(start=0, end=self.num_channels//2, dtype=torch.float32, device=x.device)
        freqs = freqs / (self.num_channels // 2 - (1 if self.endpoint else 0))
        freqs = (1 / self.max_positions) ** freqs
        x = x.ger(freqs.to(x.dtype))
        x = torch.cat([x.cos(), x.sin()], dim=1)
        return x

#----------------------------------------------------------------------------
# Timestep embedding used in the NCSN++ architecture.

class FourierEmbedding(torch.nn.Module):
    def __init__(self, num_channels, scale=16):
        super().__init__()
        self.register_buffer('freqs', torch.randn(num_channels // 2) * scale)

    def forward(self, x):
        x = x.ger((2 * np.pi * self.freqs).to(x.dtype))
        x = torch.cat([x.cos(), x.sin()], dim=1)
        return x

#----------------------------------------------------------------------------
# Reimplementation of the DDPM++ and NCSN++ architectures from the paper
# "Score-Based Generative Modeling through Stochastic Differential
# Equations". Equivalent to the original implementation by Song et al.,
# available at https://github.com/yang-song/score_sde_pytorch

class SongUNet(torch.nn.Module):
    def __init__(self,
        img_resolution,                     # Image resolution at input/output.
        in_channels,                        # Number of color channels at input.
        out_channels,                       # Number of color channels at output.
        label_dim           = 0,            # Number of class labels, 0 = unconditional.
        augment_dim         = 0,            # Augmentation label dimensionality, 0 = no augmentation.

        model_channels      = 128,          # Base multiplier for the number of channels.
        channel_mult        = [1,2,2,2],    # Per-resolution multipliers for the number of channels.
        channel_mult_emb    = 4,            # Multiplier for the dimensionality of the embedding vector.
        num_blocks          = 4,            # Number of residual blocks per resolution.
        attn_resolutions    = [16],         # List of resolutions with self-attention.
        dropout             = 0.10,         # Dropout probability of intermediate activations.
        label_dropout       = 0,            # Dropout probability of class labels for classifier-free guidance.

        embedding_type      = 'positional', # Timestep embedding type: 'positional' for DDPM++, 'fourier' for NCSN++.
        channel_mult_noise  = 1,            # Timestep embedding size: 1 for DDPM++, 2 for NCSN++.
        encoder_type        = 'standard',   # Encoder architecture: 'standard' for DDPM++, 'residual' for NCSN++.
        decoder_type        = 'standard',   # Decoder architecture: 'standard' for both DDPM++ and NCSN++.
        resample_filter     = [1,1],        # Resampling filter: [1,1] for DDPM++, [1,3,3,1] for NCSN++.
    ):
        assert embedding_type in ['fourier', 'positional']
        assert encoder_type in ['standard', 'skip', 'residual']
        assert decoder_type in ['standard', 'skip']

        super().__init__()
        self.label_dropout = label_dropout
        emb_channels = model_channels * channel_mult_emb
        noise_channels = model_channels * channel_mult_noise
        init = dict(init_mode='xavier_uniform')
        init_zero = dict(init_mode='xavier_uniform', init_weight=1e-5)
        init_attn = dict(init_mode='xavier_uniform', init_weight=np.sqrt(0.2))
        block_kwargs = dict(
            emb_channels=emb_channels, num_heads=1, dropout=dropout, skip_scale=np.sqrt(0.5), eps=1e-6,
            resample_filter=resample_filter, resample_proj=True, adaptive_scale=False,
            init=init, init_zero=init_zero, init_attn=init_attn,
        )

        # Mapping.
        self.map_noise = PositionalEmbedding(num_channels=noise_channels, endpoint=True) if embedding_type == 'positional' else FourierEmbedding(num_channels=noise_channels)
        self.map_label = Linear(in_features=label_dim, out_features=noise_channels, **init) if label_dim else None
        self.map_augment = Linear(in_features=augment_dim, out_features=noise_channels, bias=False, **init) if augment_dim else None
        self.map_layer0 = Linear(in_features=noise_channels, out_features=emb_channels, **init)
        self.map_layer1 = Linear(in_features=emb_channels, out_features=emb_channels, **init)

        # Encoder.
        self.enc = torch.nn.ModuleDict()
        cout = in_channels
        caux = in_channels
        for level, mult in enumerate(channel_mult):
            res = img_resolution >> level
            if level == 0:
                cin = cout
                cout = model_channels
                self.enc[f'{res}x{res}_conv'] = Conv2d(in_channels=cin, out_channels=cout, kernel=3, **init)
            else:
                self.enc[f'{res}x{res}_down'] = UNetBlock(in_channels=cout, out_channels=cout, down=True, **block_kwargs)
                if encoder_type == 'skip':
                    self.enc[f'{res}x{res}_aux_down'] = Conv2d(in_channels=caux, out_channels=caux, kernel=0, down=True, resample_filter=resample_filter)
                    self.enc[f'{res}x{res}_aux_skip'] = Conv2d(in_channels=caux, out_channels=cout, kernel=1, **init)
                if encoder_type == 'residual':
                    self.enc[f'{res}x{res}_aux_residual'] = Conv2d(in_channels=caux, out_channels=cout, kernel=3, down=True, resample_filter=resample_filter, fused_resample=True, **init)
                    caux = cout
            for idx in range(num_blocks):
                cin = cout
                cout = model_channels * mult
                attn = (res in attn_resolutions)
                self.enc[f'{res}x{res}_block{idx}'] = UNetBlock(in_channels=cin, out_channels=cout, attention=attn, **block_kwargs)
        skips = [block.out_channels for name, block in self.enc.items() if 'aux' not in name]

        # Decoder.
        self.dec = torch.nn.ModuleDict()
        for level, mult in reversed(list(enumerate(channel_mult))):
            res = img_resolution >> level
            if level == len(channel_mult) - 1:
                self.dec[f'{res}x{res}_in0'] = UNetBlock(in_channels=cout, out_channels=cout, attention=True, **block_kwargs)
                self.dec[f'{res}x{res}_in1'] = UNetBlock(in_channels=cout, out_channels=cout, **block_kwargs)
            else:
                self.dec[f'{res}x{res}_up'] = UNetBlock(in_channels=cout, out_channels=cout, up=True, **block_kwargs)
            for idx in range(num_blocks + 1):
                cin = cout + skips.pop()
                cout = model_channels * mult
                attn = (idx == num_blocks and res in attn_resolutions)
                self.dec[f'{res}x{res}_block{idx}'] = UNetBlock(in_channels=cin, out_channels=cout, attention=attn, **block_kwargs)
            if decoder_type == 'skip' or level == 0:
                if decoder_type == 'skip' and level < len(channel_mult) - 1:
                    self.dec[f'{res}x{res}_aux_up'] = Conv2d(in_channels=out_channels, out_channels=out_channels, kernel=0, up=True, resample_filter=resample_filter)
                self.dec[f'{res}x{res}_aux_norm'] = GroupNorm(num_channels=cout, eps=1e-6)
                self.dec[f'{res}x{res}_aux_conv'] = Conv2d(in_channels=cout, out_channels=out_channels, kernel=3, **init_zero)

    def forward(self, x, noise_labels, class_labels, augment_labels=None):
        # Mapping.
        emb = self.map_noise(noise_labels)
        emb = emb.reshape(emb.shape[0], 2, -1).flip(1).reshape(*emb.shape) # swap sin/cos
        if self.map_label is not None:
            tmp = class_labels
            if self.training and self.label_dropout:
                tmp = tmp * (torch.rand([x.shape[0], 1], device=x.device) >= self.label_dropout).to(tmp.dtype)
            emb = emb + self.map_label(tmp * np.sqrt(self.map_label.in_features))
        if self.map_augment is not None and augment_labels is not None:
            emb = emb + self.map_augment(augment_labels)
        emb = silu(self.map_layer0(emb))
        emb = silu(self.map_layer1(emb))

        # Encoder.
        skips = []
        aux = x
        for name, block in self.enc.items():
            if 'aux_down' in name:
                aux = block(aux)
            elif 'aux_skip' in name:
                x = skips[-1] = x + block(aux)
            elif 'aux_residual' in name:
                x = skips[-1] = aux = (x + block(aux)) / np.sqrt(2)
            else:
                x = block(x, emb) if isinstance(block, UNetBlock) else block(x)
                skips.append(x)

        # Decoder.
        aux = None
        tmp = None
        for name, block in self.dec.items():
            if 'aux_up' in name:
                aux = block(aux)
            elif 'aux_norm' in name:
                tmp = block(x)
            elif 'aux_conv' in name:
                tmp = block(silu(tmp))
                aux = tmp if aux is None else tmp + aux
            else:
                if x.shape[1] != block.in_channels:
                    x = torch.cat([x, skips.pop()], dim=1)
                x = block(x, emb)
        return aux

#----------------------------------------------------------------------------
class EDMPrecond(torch.nn.Module):
    def __init__(self,
        img_resolution,                     # Image resolution.
        img_channels,                       # Number of color channels.
        label_dim       = 0,                # Number of class labels, 0 = unconditional.
        use_fp16        = False,            # Execute the underlying model at FP16 precision?
        sigma_min       = 0,                # Minimum supported noise level.
        sigma_max       = float('inf'),     # Maximum supported noise level.
        sigma_data      = 0.5,              # Expected standard deviation of the training data.
        model_type      = 'DhariwalUNet',   # Class name of the underlying model.
        **model_kwargs,                     # Keyword arguments for the underlying model.
    ):
        super().__init__()
        self.img_resolution = img_resolution
        self.img_channels = img_channels
        self.label_dim = label_dim
        self.use_fp16 = use_fp16
        self.sigma_min = sigma_min
        self.sigma_max = sigma_max
        self.sigma_data = sigma_data
        self.model = globals()[model_type](img_resolution=img_resolution, in_channels=img_channels, out_channels=img_channels, label_dim=label_dim, **model_kwargs)

    def forward(self, x, sigma, class_labels=None, force_fp32=False, **model_kwargs):
        x = x.to(torch.float32)
        sigma = sigma.to(torch.float32).reshape(-1, 1, 1, 1)
        class_labels = None if self.label_dim == 0 else torch.zeros([1, self.label_dim], device=x.device) if class_labels is None else class_labels.to(torch.float32).reshape(-1, self.label_dim)
        dtype = torch.float16 if (self.use_fp16 and not force_fp32 and x.device.type == 'cuda') else torch.float32

        c_skip = self.sigma_data ** 2 / (sigma ** 2 + self.sigma_data ** 2) #1
        c_out = sigma * self.sigma_data / (sigma ** 2 + self.sigma_data ** 2).sqrt() #0
        c_in = 1 / (self.sigma_data ** 2 + sigma ** 2).sqrt()
        c_noise = sigma.log() / 4

        F_x = self.model((c_in * x).to(dtype), c_noise.flatten(), class_labels=class_labels, **model_kwargs)
        assert F_x.dtype == dtype
        D_x = c_skip * x + c_out * F_x.to(torch.float32)
        return D_x

    def round_sigma(self, sigma):
        return torch.as_tensor(sigma)

### Verification 1 — the six model sizes of Figure 18

Figure 18's legend is reproduced exactly by sweeping `model_channels` and holding every other
network argument at `main.py`'s values. This cell **asserts** it, so a silent architecture drift
cannot go unnoticed.

In [3]:
# main.py:50-57 -- the network config, with model_channels as the only free knob.
NET_KWARGS = dict(
    img_resolution   = 64,
    img_channels     = 1,
    label_dim        = 0,
    use_fp16         = False,
    model_type       = 'SongUNet',
    embedding_type   = 'positional',
    encoder_type     = 'standard',
    decoder_type     = 'standard',
    channel_mult_noise = 1,
    resample_filter  = [1, 1],
    channel_mult     = [2, 2, 2],
    dropout          = 0.0,
)
# num_blocks=4 and attn_resolutions=[16] are SongUNet defaults; main.py does not override them.

def build_net(model_channels, device=None):
    net = EDMPrecond(model_channels=model_channels, **NET_KWARGS)
    return net if device is None else net.to(device)

def count_params(net):
    return sum(p.numel() for p in net.parameters())

PAPER_FIG18_PARAMS = {4: 57017, 8: 222705, 16: 880097,
                      32: 3498945, 64: 13952897, 128: 55725825}

print(f"{'model_channels':>15} {'params':>13} {'Fig. 18 legend':>15}   match")
for C, target in PAPER_FIG18_PARAMS.items():
    n = count_params(build_net(C))
    assert n == target, f'model_channels={C}: got {n:,}, Figure 18 says {target:,}'
    print(f'{C:>15} {n:>13,} {target:>15,}   exact')
print('\nall six parameter counts reproduce Figure 18 exactly')

 model_channels        params  Fig. 18 legend   match


              4        57,017          57,017   exact
              8       222,705         222,705   exact


             16       880,097         880,097   exact


             32     3,498,945       3,498,945   exact


             64    13,952,897      13,952,897   exact


            128    55,725,825      55,725,825   exact

all six parameter counts reproduce Figure 18 exactly


### Verification 2 — the transcribed preconditioner agrees with `src/edm.py`

`src/edm.py`'s `EDMPrecond` was written independently against the EDM paper. It should be
numerically identical to the transcription above, which is a useful check in both directions: it
confirms this notebook did not mistranscribe, and it confirms the repo's own preconditioner
matches the reference implementation. The two differ only in plumbing — `src`'s takes the raw
network as a constructor argument and a 2-arg `forward`, EDM's builds the network internally and
threads `class_labels`.

If this cell ever fails, the mismatch is a real finding about `src/edm.py` and should be raised
before any result in this notebook is used.

In [4]:
import edm as src_edm   # the repo's own EDM implementation, unmodified

@torch.no_grad()
def _crosscheck_precond(model_channels=4, batch=4, seed=0):
    torch.manual_seed(seed)
    ref = build_net(model_channels).eval()          # EDM's EDMPrecond (transcribed above)

    # Wrap the *same* SongUNet instance in src/edm.py's EDMPrecond. src's calls net(x, c_noise)
    # positionally, so a 2-arg shim supplies class_labels=None.
    class _TwoArgShim(torch.nn.Module):
        def __init__(self, m):
            super().__init__(); self.m = m
        def forward(self, x, c_noise):
            return self.m(x, c_noise, class_labels=None)

    mine = src_edm.EDMPrecond(_TwoArgShim(ref.model), sigma_data=ref.sigma_data).eval()

    x = torch.randn(batch, 1, 64, 64)
    sigma = torch.tensor([0.01, 0.5, 2.0, 40.0])[:batch]
    a = ref(x, sigma)
    b = mine(x, sigma)
    return a, b

a, b = _crosscheck_precond()
max_abs = (a - b).abs().max().item()
print(f'max |EDM.EDMPrecond - src.edm.EDMPrecond| = {max_abs:.3e}  (output scale {a.abs().max():.3f})')
assert torch.allclose(a, b, rtol=0, atol=1e-6), 'src/edm.py EDMPrecond disagrees with EDM reference'
print('src/edm.py EDMPrecond matches the EDM reference')

# The EDM loss weight is also shared logic; check it against src/edm.py's helper.
_s = torch.tensor([0.01, 0.5, 2.0, 40.0])
_w_ref = (_s ** 2 + 0.5 ** 2) / (_s * 0.5) ** 2
assert torch.allclose(_w_ref, src_edm.edm_loss_weight(_s, 0.5))
print('src/edm.py edm_loss_weight matches the EDM reference')

max |EDM.EDMPrecond - src.edm.EDMPrecond| = 0.000e+00  (output scale 3.926)
src/edm.py EDMPrecond matches the EDM reference
src/edm.py edm_loss_weight matches the EDM reference


## The loss

`training/loss.py` `EDMLoss`, verbatim, at its defaults `P_mean=-1.2`, `P_std=1.2`,
`sigma_data=0.5` (`main.py:63-65` constructs it with no arguments).

Note the reduction `main.py:84` applies: `loss_fn(net, x).sum() / x.size(0)` — a **sum** over the
4096 pixels and a mean over the batch. `src/edm.py`'s `train_edm` uses a pixel *mean* instead, a
4096× smaller objective. Under Adam this is very nearly a no-op (the optimizer is scale-invariant
apart from `eps=1e-8`), but the sum is what their code does, so it is what is used here.

In [5]:
class EDMLoss:
    """training/loss.py, verbatim. Returns the per-element loss; the caller reduces it."""
    def __init__(self, P_mean=-1.2, P_std=1.2, sigma_data=0.5):
        self.P_mean = P_mean
        self.P_std = P_std
        self.sigma_data = sigma_data

    def __call__(self, net, images, labels=None, augment_pipe=None):
        rnd_normal = torch.randn([images.shape[0], 1, 1, 1], device=images.device)
        sigma = (rnd_normal * self.P_std + self.P_mean).exp()
        weight = (sigma ** 2 + self.sigma_data ** 2) / (sigma * self.sigma_data) ** 2
        y, augment_labels = augment_pipe(images) if augment_pipe is not None else (images, None)
        n = torch.randn_like(y) * sigma
        D_yn = net(y + n, sigma, labels, augment_labels=augment_labels)
        loss = weight * ((D_yn - y) ** 2)
        return loss

## The sampler

`generate.py` `edm_sampler` (EDM Algorithm 2), verbatim. `main.py:134` calls it as
`edm_sampler(ema, latents, num_steps=40)`, leaving `sigma_min=0.002`, `sigma_max=80`, `rho=7`,
`S_churn=0`, `S_min=0`, `S_max=inf`, `S_noise=1` at their defaults.

As noted at the top: `S_churn=0` makes `gamma=0`, hence `t_hat == t_cur` and
$\sqrt{t_{hat}^2 - t_{cur}^2} = 0$, so no noise is injected and this is deterministic 2nd-order
Heun — the **ODE** sampler. The paper's text describes an SDE. Both settings are available below;
the default reproduces the released code.

**The one deviation from verbatim.** EDM runs the sampler in `float64`, which MPS cannot allocate
at all. The four `torch.float64` literals are replaced by a module-level `SAMPLER_DTYPE` so the
notebook can smoke-test on an Apple laptop; it stays `float64` on CUDA and CPU and only drops to
`float32` on MPS, with a warning. **Results reported from this notebook should come from a CUDA
run**, where the dtype is exactly EDM's. Sampler precision is not something a reproduction should
quietly change, so the substitution is marked inline on every affected line.

In [6]:
# float64 everywhere, exactly as EDM -- except on MPS, which cannot allocate float64 at all.
SAMPLER_DTYPE = torch.float32 if DEVICE.type == 'mps' else torch.float64
if SAMPLER_DTYPE is torch.float32:
    print('WARNING: MPS cannot do float64; sampling in float32. EDM (and a CUDA run) uses '
          'float64 -- do not report numbers from an MPS run.')


def edm_sampler(
    net, latents, class_labels=None, randn_like=torch.randn_like,
    num_steps=18, sigma_min=0.002, sigma_max=80, rho=7,
    S_churn=0, S_min=0, S_max=float('inf'), S_noise=1,
):
    """generate.py, verbatim (EDM Algorithm 2)."""
    # Adjust noise levels based on what's supported by the network.
    sigma_min = max(sigma_min, net.sigma_min)
    sigma_max = min(sigma_max, net.sigma_max)

    # Time step discretization.
    step_indices = torch.arange(num_steps, dtype=SAMPLER_DTYPE, device=latents.device)  # EDM: torch.float64
    t_steps = (sigma_max ** (1 / rho) + step_indices / (num_steps - 1) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))) ** rho
    t_steps = torch.cat([net.round_sigma(t_steps), torch.zeros_like(t_steps[:1])]) # t_N = 0

    # Main sampling loop.
    x_next = latents.to(SAMPLER_DTYPE) * t_steps[0]  # EDM: torch.float64
    for i, (t_cur, t_next) in enumerate(zip(t_steps[:-1], t_steps[1:])): # 0, ..., N-1
        x_cur = x_next

        # Increase noise temporarily.
        gamma = min(S_churn / num_steps, np.sqrt(2) - 1) if S_min <= t_cur <= S_max else 0
        t_hat = net.round_sigma(t_cur + gamma * t_cur)
        x_hat = x_cur + (t_hat ** 2 - t_cur ** 2).sqrt() * S_noise * randn_like(x_cur)

        # Euler step.
        denoised = net(x_hat, t_hat, class_labels).to(SAMPLER_DTYPE)  # EDM: torch.float64
        d_cur = (x_hat - denoised) / t_hat
        x_next = x_hat + (t_next - t_hat) * d_cur

        # Apply 2nd order correction.
        if i < num_steps - 1:
            denoised = net(x_next, t_next, class_labels).to(SAMPLER_DTYPE)  # EDM: torch.float64
            d_prime = (x_next - denoised) / t_next
            x_next = x_hat + (t_next - t_hat) * (0.5 * d_cur + 0.5 * d_prime)

    return x_next

## The training data

`main.py:26-29`, verbatim:

```python
data = torch.zeros((Ntrain_samps,1,64,64))
data[0,0,6:18,6:18] = 1.0
data[1,0,40:54,40:54] = 1.0
```

So the two squares are **not** the same size: a $12\times12$ square inset 6 pixels from the
top-left corner, and a $14\times14$ square inset 10 pixels from the bottom-right. (Measuring
Figure 16's rendered pixels independently gives $12.2\times12.2$ at offset $(5.5, 5.7)$ and
$14.2\times14.1$ at offset $(39.6, 40.2)$ — consistent, so the figure and the code agree.)

The paper describes the images as binary with $d = 4096$; note that the data are **not**
zero-centred (mean ≈ 0.043) and their per-pixel std ≈ 0.20, while EDM's preconditioning is
derived assuming data with std $\sigma_{data} = 0.5$. `main.py` leaves `sigma_data` at the 0.5
default rather than estimating it, so that mismatch is part of their setup and is reproduced.

In [7]:
# main.py:26-29, verbatim
N_TRAIN = 2
data = torch.zeros((N_TRAIN, 1, 64, 64))
data[0, 0, 6:18, 6:18] = 1.0
data[1, 0, 40:54, 40:54] = 1.0

flat = data.reshape(N_TRAIN, -1)
pair_dist = torch.cdist(flat, flat)[0, 1].item()
print(f'values          : {sorted(set(data.unique().tolist()))}')
print(f'squares         : {int(data[0].sum())} px (12x12) and {int(data[1].sum())} px (14x14)')
print(f'mean / std      : {data.mean():.4f} / {data.std():.4f}   (sigma_data is held at 0.5)')
print(f'||x0||          : {flat.norm(dim=1).tolist()}')
print(f'D (pair spacing): {pair_dist:.4f}   -> sampler sigma_max/D = {80.0 / pair_dist:.3f}')

# Figure 16
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
for j in range(N_TRAIN):
    axes[j].imshow(data[j, 0], vmin=0, vmax=1)
    axes[j].set_xticks([]); axes[j].set_yticks([])
fig.suptitle('Figure 16 — training data for the rectangles dataset')
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'baptista_rect_fig16_training_data.png'), dpi=150,
            bbox_inches='tight')
plt.show()

values          : [0.0, 1.0]
squares         : 144 px (12x12) and 196 px (14x14)
mean / std      : 0.0415 / 0.1995   (sigma_data is held at 0.5)
||x0||          : [12.0, 14.0]
D (pair spacing): 18.4391   -> sampler sigma_max/D = 4.339


## The memorization metric

§5.4, verbatim:

> *"After every 1000 optimization steps, we generate 100 samples from each model and compute the
> proportion of them that exactly match one of the $N = 2$ training points, i.e., the fraction of
> samples at a Euclidean distance of zero from $x_0^n$ for some $n$. To avoid small noise errors,
> we take all the generated samples and do a thresholding where every value of the generated image
> above 0.5 is mapped to 1 and every value below 0.5 is mapped to 0."*

So: threshold at 0.5, then require distance **exactly** zero. On binarized images that is the same
as "zero mismatched pixels", which is what is computed here (integer comparison, no floating-point
knife edge).

`n_mismatch` — the number of differing pixels to the nearest training image — is recorded
alongside, because a hard 0/1 indicator tells you nothing about *how close* a near-miss was, and
the distinction between "1 pixel off" and "800 pixels off" is exactly what distinguishes a model
about to memorize from one that is nowhere near.

In [8]:
@torch.no_grad()
def collapse_stats(x_gen, x_train, threshold=0.5):
    """Baptista et al. §5.4 collapse metric, plus a mismatched-pixel diagnostic.

    x_gen:   (G, 1, 64, 64) generated images, unthresholded
    x_train: (N, 1, 64, 64) binary training images
    """
    g = (x_gen.detach().float().cpu() > threshold).reshape(x_gen.shape[0], -1)
    t = (x_train.detach().float().cpu() > threshold).reshape(x_train.shape[0], -1)
    # (G, N) count of differing pixels
    mism = (g[:, None, :] != t[None, :, :]).sum(dim=2)
    n_mismatch = mism.min(dim=1).values           # to the nearest training image
    return {
        'fraction': (n_mismatch == 0).float().mean().item(),   # the paper's number
        'n_mismatch_median': n_mismatch.median().item(),
        'n_mismatch_min': n_mismatch.min().item(),
        'nn_index': mism.argmin(dim=1),
    }

# sanity: the training images score 1.0 against themselves, and noise scores 0.0
assert collapse_stats(data, data)['fraction'] == 1.0
assert collapse_stats(torch.rand(64, 1, 64, 64), data)['fraction'] == 0.0
print('collapse metric: training data -> 1.00, uniform noise -> 0.00')

collapse metric: training data -> 1.00, uniform noise -> 0.00


## Configuration

Every value below is either taken from `main.py` (marked) or from the §5.4 prose for the
Figure-18 protocol, whose script was not released.

### The epoch / optimization-step ambiguity

`main.py` runs `for ep in range(50000)` with a `batch_size=1` loader over $N=2$ images, so one
"epoch" is **two** optimizer updates. Figure 17 is captioned in *epochs* and runs to 50,000;
Figure 18's x-axis is labelled *"Optimization steps"* and also runs to 50,000. Since the released
script's only loop counter is `ep`, the two axes are almost certainly the same quantity, i.e. one
pass over the 2-image dataset — which the prose calls "a batch size of 2".

This notebook takes that reading: **`eval_every` and `epochs` are counted in passes over the
dataset**, so the x-axis is directly comparable to Figures 17 and 18. All three counters
(`epoch`, `opt_step`, `cur_nimg`) are recorded at every evaluation, so the curve can be re-plotted
against optimizer updates instead without re-running anything.

In [9]:
SMOKE = False    # True -> a few-minute end-to-end check of the whole pipeline

CFG = dict(
    # -- from main.py, verbatim --------------------------------------------------------------
    epochs             = 50_000,   # main.py:15
    batch_mode         = 'main_py',# 'main_py'  : DataLoader(batch_size=1, shuffle=True), 2 updates/epoch
                                   # 'paper_bs2': one update/epoch on a batch of 2 (what the prose says)
    lr                 = 10e-4,    # main.py:61
    betas              = (0.9, 0.999),
    eps                = 1e-8,
    lr_rampup_kimg     = 10_000,   # main.py:19  -> lr never exceeds 1e-5 over this run
    ema_halflife_kimg  = 500,      # main.py:20
    ema_rampup_ratio   = 0.05,     # main.py:21
    P_mean             = -1.2,     # EDMLoss defaults, main.py:63-65
    P_std              = 1.2,
    sigma_data         = 0.5,      # networks.py:639 default; NOT estimated from the data

    # -- sampler: main.py:134 calls edm_sampler(ema, latents, num_steps=40), rest defaulted ----
    num_steps          = 40,
    sigma_min          = 0.002,
    sigma_max          = 80.0,
    rho                = 7,
    S_churn            = 0.0,      # released default -> deterministic Heun (ODE). See CFG_SDE_ALT.
    S_min              = 0.0,
    S_max              = float('inf'),
    S_noise            = 1.0,

    # -- Figure 18 protocol: from the paper text (script not released) -------------------------
    model_channels_sweep = [4, 8, 16, 32, 64, 128],
    n_eval_samples     = 100,      # "we generate 100 samples from each model"
    eval_every         = 1_000,    # "after every 1000 optimization steps"
    threshold          = 0.5,      # "every value above 0.5 is mapped to 1"

    # -- run mechanics (not from the paper) ----------------------------------------------------
    seed               = 0,
    latent_seed        = 42,       # eval latents are seeded per evaluation for reproducibility
    eval_batch         = 50,       # lower this first if the 55.7M arm runs out of GPU memory
    save_resume_state  = True,     # net+ema+optimizer dumped each eval so a SLURM timeout resumes
    n_sample_grid      = 16,       # Figure 17 shows 16 samples
)

# Karras et al.'s CIFAR-10 stochastic preset, for the SDE reading of the paper text.
# Apply with: CFG.update(CFG_SDE_ALT)
CFG_SDE_ALT = dict(S_churn=30.0, S_min=0.01, S_max=1.0, S_noise=1.007)

if SMOKE:
    CFG.update(epochs=300, eval_every=100, n_eval_samples=16, num_steps=10,
               model_channels_sweep=[4, 8], eval_batch=16, save_resume_state=False)

# Arm selection can be overridden from the environment, so a SLURM array job can put one
# model size on each GPU:  RECT_ARMS=128 sbatch ...   /   RECT_ARMS='4 8' python ...
_arms_env = os.environ.get('RECT_ARMS')
if _arms_env:
    CFG['model_channels_sweep'] = [int(c) for c in _arms_env.replace(',', ' ').split()]
    print(f'RECT_ARMS override -> {CFG["model_channels_sweep"]}')

# One file per arm, never a shared one: concurrent array tasks would otherwise clobber
# each other's results. The plotting cells glob whatever arms are present.
RESULT_DIR = os.path.join(results_dir, 'baptista_rectangles_n2')
os.makedirs(RESULT_DIR, exist_ok=True)
STATE_DIR = RESULT_DIR

def arm_result_path(C):
    return os.path.join(RESULT_DIR, f'arm_c{C}_result.pt')

def load_runs():
    """Collect every completed arm on disk. Safe to call in a fresh kernel after array jobs."""
    out = {}
    for C in PAPER_FIG18_PARAMS:
        p = arm_result_path(C)
        if os.path.exists(p):
            out[C] = torch.load(p, map_location='cpu', weights_only=False)
    return out

n_evals = CFG['epochs'] // CFG['eval_every']
print(f"{'SMOKE RUN' if SMOKE else 'FULL RUN'}")
print(f"  arms          : {CFG['model_channels_sweep']}  "
      f"({[f'{PAPER_FIG18_PARAMS.get(c, count_params(build_net(c))):,}' for c in CFG['model_channels_sweep']]} params)")
print(f"  epochs        : {CFG['epochs']:,}  (batch_mode={CFG['batch_mode']}, "
      f"{CFG['epochs'] * (N_TRAIN if CFG['batch_mode'] == 'main_py' else 1):,} optimizer updates)")
print(f"  evaluations   : {n_evals} x {CFG['n_eval_samples']} samples at {CFG['num_steps']} sampler steps")
print(f"  sampler       : {'ODE (deterministic Heun)' if CFG['S_churn'] == 0 else 'SDE (S_churn=%g)' % CFG['S_churn']}"
      f", sigma in [{CFG['sigma_min']}, {CFG['sigma_max']}], rho={CFG['rho']}")
print(f"  results dir   : {RESULT_DIR}")
print(f"  one file per arm: arm_c<C>_result.pt (+ arm_c<C>.pt resume state)")

RECT_ARMS override -> [4]
FULL RUN


  arms          : [4]  (['57,017'] params)
  epochs        : 50,000  (batch_mode=main_py, 100,000 optimizer updates)
  evaluations   : 50 x 100 samples at 40 sampler steps
  sampler       : ODE (deterministic Heun), sigma in [0.002, 80.0], rho=7
  results dir   : /home/ishiyer/projects/aip-baptista/ishiyer/research-diffusion/results/data/baptista_rectangles_n2
  one file per arm: arm_c<C>_result.pt (+ arm_c<C>.pt resume state)


## Training

The loop below is a line-for-line transcription of `main.py:73-107` — the same ordering of
`zero_grad` → forward → `backward` → learning-rate assignment → gradient `nan_to_num` →
`optimizer.step()` → EMA update → `cur_nimg` increment. The only additions are the periodic
evaluation (Figure 18's protocol), incremental artifact saving, and resume support, none of which
touch the optimization path.

`main.py` writes a checkpoint every 100 epochs and never resumes; on a cluster with a wall-clock
limit that is not enough, so a full `net`/`ema`/`optimizer` state dump is written at each
evaluation and the arm restarts from it. The CPU, CUDA and MPS RNG states are saved with it, so a
resumed run continues the same random stream as an uninterrupted one — the loader's shuffle order
and the `EDMLoss` sigma/noise draws all pick up where they stopped. (Restarting can still shift
the last ulp of a weight, because `cudnn.benchmark = True` may select a different convolution
algorithm in the new process; the sampled trajectories and the collapse metric are unaffected.)
For the 55.7M arm that file is
~0.9 GB (weights + EMA + Adam moments), overwritten in place — set
`CFG['save_resume_state'] = False` if the disk quota matters more than restartability.

In [10]:
@torch.no_grad()
def generate_samples(ema_net, n_samples, cfg, device, seed):
    """Draw n_samples with the EDM sampler, chunked to fit in memory. main.py samples from EMA."""
    ema_net.eval()
    out = []
    remaining = n_samples
    chunk_i = 0
    while remaining > 0:
        b = min(cfg['eval_batch'], remaining)
        g = torch.Generator().manual_seed(seed + 1000 * chunk_i)
        latents = torch.randn(b, 1, 64, 64, generator=g).to(device)
        x = edm_sampler(ema_net, latents,
                        num_steps=cfg['num_steps'], sigma_min=cfg['sigma_min'],
                        sigma_max=cfg['sigma_max'], rho=cfg['rho'], S_churn=cfg['S_churn'],
                        S_min=cfg['S_min'], S_max=cfg['S_max'], S_noise=cfg['S_noise'])
        out.append(x.float().cpu())
        remaining -= b
        chunk_i += 1
    return torch.cat(out, dim=0)


def run_arm(model_channels, cfg, device, resume=True, log=print):
    """One model size, transcribing main.py's training loop. Returns the evaluation log."""
    state_path = os.path.join(STATE_DIR, f'arm_c{model_channels}.pt')

    torch.manual_seed(cfg['seed'])
    net = build_net(model_channels, device)
    net.train().requires_grad_(True)
    ema = copy.deepcopy(net).eval().requires_grad_(False)
    optimizer = torch.optim.Adam(net.parameters(), lr=cfg['lr'],
                                 betas=list(cfg['betas']), eps=cfg['eps'])
    loss_fn = EDMLoss(P_mean=cfg['P_mean'], P_std=cfg['P_std'], sigma_data=cfg['sigma_data'])

    cur_nimg, opt_step, start_epoch = 1, 0, 0     # main.py:18 starts cur_nimg at 1
    eval_log, loss_hist = [], []

    if resume and os.path.exists(state_path):
        # to CPU: load_state_dict re-places model/optimizer tensors onto the params'
        # device by itself, and the saved RNG state must stay a CPU ByteTensor.
        st = torch.load(state_path, map_location='cpu', weights_only=False)
        net.load_state_dict(st['net']); ema.load_state_dict(st['ema'])
        optimizer.load_state_dict(st['opt'])
        cur_nimg, opt_step, start_epoch = st['cur_nimg'], st['opt_step'], st['epoch']
        eval_log, loss_hist = st['eval_log'], st['loss_hist']
        # Restore the RNG too, so a resumed run is bit-identical to an uninterrupted one:
        # the loader's shuffle order, the EDMLoss sigma draws and its noise all come from here.
        torch.set_rng_state(st['rng_cpu'])
        if st.get('rng_cuda') is not None and torch.cuda.is_available():
            torch.cuda.set_rng_state_all(st['rng_cuda'])
        if st.get('rng_mps') is not None and torch.backends.mps.is_available():
            torch.mps.set_rng_state(st['rng_mps'])
        log(f'  resumed from epoch {start_epoch:,}')

    x_train_cpu = data
    if cfg['batch_mode'] == 'main_py':
        loader = torch.utils.data.DataLoader(
            torch.utils.data.TensorDataset(data), batch_size=1, shuffle=True)   # main.py:37
        def epoch_batches():
            for (x,) in loader:
                yield x
    elif cfg['batch_mode'] == 'paper_bs2':
        def epoch_batches():
            yield data[torch.randperm(N_TRAIN)]
    else:
        raise ValueError(cfg['batch_mode'])

    t_start = time.time()
    for ep in range(start_epoch, cfg['epochs']):
        err, count = 0.0, 0
        net.train()
        for x in epoch_batches():
            optimizer.zero_grad(set_to_none=True)
            x = x.to(device)
            loss = loss_fn(net, x).sum() / x.size(0)          # main.py:84
            loss.backward()

            for g in optimizer.param_groups:                   # main.py:88-89
                g['lr'] = cfg['lr'] * min(cur_nimg / max(cfg['lr_rampup_kimg'] * 1000, 1e-8), 1)
            for param in net.parameters():                     # main.py:91-93
                if param.grad is not None:
                    torch.nan_to_num(param.grad, nan=0, posinf=1e5, neginf=-1e5, out=param.grad)
            optimizer.step()

            ema_halflife_nimg = cfg['ema_halflife_kimg'] * 1000        # main.py:96-102
            if cfg['ema_rampup_ratio'] is not None:
                ema_halflife_nimg = min(ema_halflife_nimg, cur_nimg * cfg['ema_rampup_ratio'])
            ema_beta = 0.5 ** (x.size(0) / max(ema_halflife_nimg, 1e-8))
            for p_ema, p_net in zip(ema.parameters(), net.parameters()):
                p_ema.copy_(p_net.detach().lerp(p_ema, ema_beta))

            err += loss.item()
            cur_nimg += x.size(0)
            count += x.size(0)
            opt_step += 1

        loss_hist.append(err / count)

        if (ep + 1) % cfg['eval_every'] == 0:
            n_done = (ep + 1) // cfg['eval_every']
            x_gen = generate_samples(ema, cfg['n_eval_samples'], cfg, device,
                                     seed=cfg['latent_seed'] + n_done)
            st = collapse_stats(x_gen, x_train_cpu, threshold=cfg['threshold'])
            row = dict(epoch=ep + 1, opt_step=opt_step, cur_nimg=cur_nimg,
                       lr=optimizer.param_groups[0]['lr'],
                       loss=float(np.mean(loss_hist[-cfg['eval_every']:])),
                       fraction=st['fraction'],
                       n_mismatch_median=int(st['n_mismatch_median']),
                       n_mismatch_min=int(st['n_mismatch_min']),
                       samples=x_gen[:cfg['n_sample_grid']].clone())
            eval_log.append(row)

            el = time.time() - t_start
            frac_done = (ep + 1 - start_epoch) / max(cfg['epochs'] - start_epoch, 1)
            eta = el / max(frac_done, 1e-9) - el
            log(f"  epoch {ep+1:>7,} | step {opt_step:>7,} | lr {row['lr']:.2e} | "
                f"loss {row['loss']:.4f} | collapse {st['fraction']:.2f} | "
                f"mismatch(med) {int(st['n_mismatch_median']):>4d} px | "
                f"{el/60:.1f}m elapsed, ~{eta/60:.0f}m left")

            if cfg['save_resume_state']:
                torch.save(dict(net=net.state_dict(), ema=ema.state_dict(),
                                opt=optimizer.state_dict(), cur_nimg=cur_nimg,
                                opt_step=opt_step, epoch=ep + 1,
                                eval_log=eval_log, loss_hist=loss_hist,
                                rng_cpu=torch.get_rng_state(),
                                rng_cuda=(torch.cuda.get_rng_state_all()
                                          if torch.cuda.is_available() else None),
                                rng_mps=(torch.mps.get_rng_state()
                                         if torch.backends.mps.is_available() else None)),
                           state_path)

    return dict(eval_log=eval_log, loss_hist=loss_hist,
                n_params=count_params(net), model_channels=model_channels,
                minutes=(time.time() - t_start) / 60)

### Run the sweep

Arms run largest-last so the cheap ones establish the trend before the expensive one starts.
Completed arms are skipped on re-run, and a partially-finished arm resumes from its state dump,
so this cell is safe to re-execute after a wall-clock kill.

Under SLURM, run it headless with:

```bash
python3 -m nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 --ExecutePreprocessor.kernel_name=python3 notebooks/multiscale/baptista_rectangles_n2_reproduction.ipynb
```

Because `--inplace` only writes outputs at the very end, watch progress through the SLURM log or
the incremental `.pt` artifact, not the notebook.

In [11]:
NOTE = ('Baptista et al. arXiv:2501.15785 section 5.4, N=2 rectangles. Config transcribed '
        'from baptistar/DiffusionModelDynamics RectangleImages/main.py; Figure 18 protocol '
        'from the paper text. Sampler is deterministic Heun at S_churn=0 (the released '
        'default) unless CFG_SDE_ALT was applied.')

n_expect = CFG['epochs'] // CFG['eval_every']
for C in sorted(CFG['model_channels_sweep']):
    p = arm_result_path(C)
    if os.path.exists(p):
        prev = torch.load(p, map_location='cpu', weights_only=False)
        if len(prev['eval_log']) >= n_expect:
            print(f'=== model_channels={C} already complete, skipping ===', flush=True)
            continue
    n_par = count_params(build_net(C))
    print(f'=== model_channels={C}  ({n_par:,} params) ===', flush=True)
    res = run_arm(C, CFG, DEVICE, resume=True)
    res.update(cfg={k: v for k, v in CFG.items()}, data=data, note=NOTE)
    torch.save(res, p)
    print(f"  done in {res['minutes']:.1f} min -> {p}", flush=True)

runs = load_runs()
print('\narms on disk: ' + ', '.join('C%d=%s params' % (C, format(runs[C]['n_params'], ','))
                                     for C in sorted(runs)))

=== model_channels=4  (57,017 params) ===


  epoch   1,000 | step   2,000 | lr 2.00e-07 | loss 2816.2292 | collapse 0.00 | mismatch(med)  755 px | 2.7m elapsed, ~132m left


  epoch   2,000 | step   4,000 | lr 4.00e-07 | loss 2831.3704 | collapse 0.00 | mismatch(med)  763 px | 5.3m elapsed, ~126m left


  epoch   3,000 | step   6,000 | lr 6.00e-07 | loss 2767.5375 | collapse 0.00 | mismatch(med)  768 px | 7.8m elapsed, ~122m left


  epoch   4,000 | step   8,000 | lr 8.00e-07 | loss 2746.0076 | collapse 0.00 | mismatch(med)  784 px | 10.4m elapsed, ~119m left


...

  epoch  47,000 | step  94,000 | lr 9.40e-06 | loss 70.0629 | collapse 0.96 | mismatch(med)    0 px | 120.6m elapsed, ~8m left


  epoch  48,000 | step  96,000 | lr 9.60e-06 | loss 65.4813 | collapse 0.96 | mismatch(med)    0 px | 123.1m elapsed, ~5m left


  epoch  49,000 | step  98,000 | lr 9.80e-06 | loss 69.5133 | collapse 0.96 | mismatch(med)    0 px | 125.7m elapsed, ~3m left


  epoch  50,000 | step 100,000 | lr 1.00e-05 | loss 65.9412 | collapse 0.97 | mismatch(med)    0 px | 128.3m elapsed, ~0m left
  done in 128.3 min -> /home/ishiyer/projects/aip-baptista/ishiyer/research-diffusion/results/data/baptista_rectangles_n2/arm_c4_result.pt



arms on disk: C4=57,017 params, C8=222,705 params, C16=880,097 params


## Result — Figure 18 (left, $N=2$)

The claim being tested: *"all models transition to perfect memorization if trained for a
sufficient amount of time. However, increasing the number of model parameters results in this
occurring faster during the optimization."*

In [12]:
runs = load_runs()          # works in a fresh kernel after SLURM array jobs
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
cmap = plt.get_cmap('tab10')
order = sorted(runs.keys())

for i, C in enumerate(order):
    r = runs[C]
    ep = [e['epoch'] for e in r['eval_log']]
    fr = [e['fraction'] for e in r['eval_log']]
    mm = [e['n_mismatch_median'] for e in r['eval_log']]
    axes[0].plot(ep, fr, color=cmap(i), lw=1.5, label=f"{r['n_params']:,}")
    axes[1].plot(ep, np.maximum(mm, 0.5), color=cmap(i), lw=1.5, label=f"{r['n_params']:,}")

axes[0].set_xlabel('Optimization steps')
axes[0].set_ylabel('Fraction of samples matching data')
axes[0].set_ylim(-0.02, 1.02)
axes[0].set_title('Figure 18 (left), $N=2$ — reproduction')
axes[0].legend(fontsize=8, title='parameters', title_fontsize=8)

axes[1].set_yscale('log')
axes[1].set_xlabel('Optimization steps')
axes[1].set_ylabel('median mismatched pixels to nearest train (clipped at 0.5)')
axes[1].set_title('how close the near-misses are')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'baptista_rect_fig18_n2.png'), dpi=150, bbox_inches='tight')
plt.show()

In [13]:
# The gate, and the ordering claim.
print(f"{'params':>12} {'model_ch':>9} {'peak frac':>10} {'first step >=0.9':>17} {'final frac':>11}")
print('-' * 64)
onsets = {}
for C in order:
    r = runs[C]
    fr = [e['fraction'] for e in r['eval_log']]
    ep = [e['epoch'] for e in r['eval_log']]
    hit = next((ep[i] for i, v in enumerate(fr) if v >= 0.9), None)
    onsets[C] = hit
    print(f"{r['n_params']:>12,} {C:>9} {max(fr):>10.2f} "
          f"{(f'{hit:,}' if hit else 'not reached'):>17} {fr[-1]:>11.2f}")
print('-' * 64)

reached = [C for C in order if onsets[C] is not None]
print(f'GATE  : {len(reached)}/{len(order)} arms reach collapse fraction >= 0.9')
if len(reached) == len(order):
    print('        -> matches "all models transition to perfect memorization"')
else:
    missing = [f'{runs[C]["n_params"]:,}' for C in order if onsets[C] is None]
    print(f'        -> arms that did NOT memorize: {missing}')
    print('        Check, in order: (1) sampler -- S_churn=0 gives the ODE, try CFG_SDE_ALT;')
    print('        (2) the lr ramp-up, which caps the effective lr at ~1e-5 over this budget;')
    print('        (3) batch_mode; (4) more epochs.')

if len(reached) >= 2:
    mono = all(onsets[reached[i]] >= onsets[reached[i + 1]] for i in range(len(reached) - 1))
    print(f'ORDER : memorization onset is monotone decreasing in model size: {mono}')
    print('        -> matches "increasing the number of model parameters results in this '
          'occurring faster"' if mono else '        -> does NOT reproduce the ordering claim')

      params  model_ch  peak frac  first step >=0.9  final frac
----------------------------------------------------------------
      57,017         4       0.97            37,000        0.97
     222,705         8       1.00            22,000        0.98
     880,097        16       1.00            14,000        0.98
----------------------------------------------------------------
GATE  : 3/3 arms reach collapse fraction >= 0.9
        -> matches "all models transition to perfect memorization"
ORDER : memorization onset is monotone decreasing in model size: True
        -> matches "increasing the number of model parameters results in this occurring faster"


## Result — Figure 17

Sixteen generated samples as training proceeds, for the largest arm (`model_channels=128`, which
is `main.py`'s own configuration). Their Figure 17 shows novel-but-plausible images at 6k–20k
epochs and complete collapse onto the two training images by 50k.

In [14]:
C_show = max(runs) if runs else None
if C_show is not None:
    r = runs[C_show]
    log_rows = r['eval_log']
    # pick evaluation points nearest the epochs shown in their Figure 17
    want = [2000, 4000, 6000, 10000, 20000, 50000]
    avail = [e['epoch'] for e in log_rows]
    picks, seen = [], set()
    for w in want:
        if not avail:
            break
        j = int(np.argmin([abs(a - w) for a in avail]))
        if j not in seen:
            seen.add(j); picks.append(j)

    n_show = min(CFG['n_sample_grid'], log_rows[0]['samples'].shape[0])
    side = int(math.ceil(math.sqrt(n_show)))
    fig, axes = plt.subplots(len(picks), n_show, figsize=(0.75 * n_show, 0.95 * len(picks)),
                             squeeze=False)
    for row, j in enumerate(picks):
        s = log_rows[j]['samples']
        for k in range(n_show):
            axes[row][k].imshow(s[k, 0], vmin=0, vmax=1)
            axes[row][k].set_xticks([]); axes[row][k].set_yticks([])
        axes[row][0].set_ylabel(f"{log_rows[j]['epoch']//1000}k\n{log_rows[j]['fraction']:.2f}",
                                fontsize=7, rotation=0, ha='right', va='center')
    fig.suptitle(f'Figure 17 — samples vs training time  '
                 f'(model_channels={C_show}, {r["n_params"]:,} params)\n'
                 f'row label: epochs / collapse fraction', fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, 'baptista_rect_fig17_samples.png'), dpi=150,
                bbox_inches='tight')
    plt.show()

## Notes

**What a pass licenses.** If every arm reaches a collapse fraction near 1.0 and the larger models
get there sooner, this transcription of their pipeline memorizes exactly as reported. That makes
it a working positive control: our multiband Matérn U-Net result — no memorization — is then a
statement about the *data* and the model's inductive bias on it, not a broken training loop,
sampler, or metric. That is the only inference it supports. It says nothing about which of the two
datasets' properties is responsible.

**What a failure means.** A failure here is a failure of *this notebook*, not evidence against
their result, and it must be debugged before it is reported anywhere. The likely causes, in the
order worth checking, are the three flagged at the top: the `S_churn=0` ODE/SDE discrepancy, the
learning-rate ramp-up that caps the effective rate near $10^{-5}$, and the `batch_size=1` loader.

**Two settings deliberately differ from this repo's locked conventions**, because this reproduces
their experiment rather than extending ours:

* $\sigma_{max} = 80$ with EDM's $\rho=7$ Karras schedule at 40 steps, not our locked
  $\sigma_{max} = 10$ / 1000-step Euler–Maruyama. Their spacing is $D = \|x_0^1 - x_0^2\|
  \approx 18.4$, so $\sigma_{max}/D \approx 4.3$ — the sampler starts well above the scale that
  separates the two training points.
* the metric is their pixel-space thresholded exact match, not our per-band ring ratio, and
  `exclude_nn` has no analogue here.

Numbers from this notebook therefore belong in their own figure with the difference stated in the
caption, and must not be tabulated next to `edm_unet_*` results.

**Single seed.** Every arm runs at `seed=0`. The onset step is visibly noisy in their own
Figure 18 (curves are non-monotone and cross), so a small difference in onset between two adjacent
model sizes should not be read as a capacity effect without repeat seeds.

**If you want the SDE.** `CFG.update(CFG_SDE_ALT)` before the run cell switches to Karras's
CIFAR-10 stochastic preset. Running both is the clean way to settle whether their §5.4 SDE claim
and their released `S_churn=0` code give the same answer — worth reporting either way, and worth
raising with Prof. Baptista since it is a discrepancy in their own material.